In [0]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

CATALOG      = "sample_project_playground_003"
SOURCE_TABLE = f"{CATALOG}.bronze.entsoe_day_ahead_prices_se"
TARGET_TABLE = f"{CATALOG}.silver.entsoe_price_silver"
WINDOW_COL   = "delivery_date_local"
OVERLAP_DAYS = 7
KEYS = ["area_eic", "interval_start_utc"]

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

dbutils.widgets.dropdown("mode", "incremental", ["incremental", "full"])
mode = dbutils.widgets.get("mode")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
  delivery_date_local DATE,
  bidding_zone        STRING,
  area_eic            STRING,
  contract_type       STRING,
  resolution          STRING,
  currency            STRING,
  price_eur_mwh       DOUBLE,
  interval_start_utc  TIMESTAMP,
  interval_end_utc    TIMESTAMP,
  silver_updated_at   TIMESTAMP
)
USING DELTA
PARTITIONED BY (delivery_date_local)
""")

def merge_into(table, df, keys):
    df.createOrReplaceTempView("stg")
    on = " AND ".join(f"t.{k} = s.{k}" for k in keys)
    spark.sql(f"""
      MERGE INTO {table} t
      USING stg s
      ON {on}
      WHEN MATCHED THEN UPDATE SET *
      WHEN NOT MATCHED THEN INSERT *
    """)

# --- how much bronze to read? --------------------------------------------------
# first run (silver empty) -> ALL of bronze, so history is never missed
# later runs               -> from the last date already loaded, with overlap
# mode=full                -> ignore watermark, re-read everything (repair)
last_loaded = spark.table(TARGET_TABLE).agg(F.max(WINDOW_COL)).first()[0]

if mode == "full" or last_loaded is None:
    src = spark.table(SOURCE_TABLE)
    print("[silver:entsoe] reading ALL bronze rows (first run or mode=full)")
else:
    src = (spark.table(SOURCE_TABLE)
           .where(F.col(WINDOW_COL) >= F.date_sub(F.lit(last_loaded), OVERLAP_DAYS)))
    print(f"[silver:entsoe] incremental: bronze from {last_loaded} (-{OVERLAP_DAYS}d overlap)")

# --- keep only the columns we want -------------------------------------------
out = (src
    .select(
        F.col("delivery_date_local"),
        F.col("bidding_zone"),
        F.col("area_eic"),
        F.col("contract_type"),
        F.col("resolution"),
        F.col("currency"),
        F.col("price_eur_mwh").cast("double"),
        F.col("interval_start_utc"),
        F.col("interval_end_utc"),
        F.current_timestamp().alias("silver_updated_at"),
    )
    .dropDuplicates(KEYS)
    .cache())

# --- data quality: fail loudly BEFORE writing ---------------------------------
n = out.count()
assert n > 0, "[silver:entsoe] no rows read from bronze - check bronze table"

for k in KEYS:
    bad = out.where(F.col(k).isNull()).count()
    assert bad == 0, f"[silver:entsoe] {bad} rows with null in key {k}"

warn_price = out.where(
    F.col("price_eur_mwh").isNull()
    | (F.col("price_eur_mwh") < -500)
    | (F.col("price_eur_mwh") > 500)
).count()
if warn_price > 0:
    print(f"WARN [silver:entsoe] {warn_price} rows with price null or outside [-500,500]")

# --- write --------------------------------------------------------------------
merge_into(TARGET_TABLE, out, KEYS)
print(f"[silver:entsoe] processed={n}, total={spark.table(TARGET_TABLE).count()}")
out.unpersist()